# B2 · Tokenisation et embeddings — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/b2_tokenisation/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou des tests). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`. Le notebook télécharge les fichiers de six tokenizers du Hugging Face Hub (quelques Mo, pas les poids des modèles) s'ils ne sont pas déjà là ; sans connexion, leurs cellules affichent ⏳ et le reste continue. Tout tourne sur CPU.

> Travaille dans **ta copie** (`mon_travail/b2_tokenisation/03_notebook.ipynb`, créée par `python tools/start_chapter.py B2`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | B2.Q1 à B2.Q12, B2.R1 à B2.R3, B2.1 à B2.6, B2.9 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | B2.13 | Tokenizers réels : GPT-2, BERT et un tokenizer multilingue sur Holmes et Verne | 📦 | ★★ | 20 |
|  | B2.14 | Prédire : quel texte coûte le plus de tokens, Holmes, Verne, des nombres ou des emoji ? | 🔮 | ★ | 10 |
| B | B2.15 | get_pair_counts et merge_pair | 🔨 | ★★ | 20 |
|  | B2.16 | BPETokenizer.fit : apprendre les fusions, au niveau caractère puis octet | 🔨 | ★★★ | 50 |
|  | B2.17 | encode et decode : fusions appliquées par rang, aller-retour sans perte | 🔨 | ★★★ | 40 |
|  | B2.18 | Le tokenizer qui abîme les accents : trois bugs d'Unicode et de fusions | 🐛 | ★★ | 25 |
|  | B2.19 | Ton BPE contre la bibliothèque tokenizers : mêmes fusions, autre vitesse | 📦 | ★★ | 25 |
| C | B2.20 | Taille du vocabulaire : compression et entropie par token, de 300 à 8 000 tokens | 🔬 | ★★★ | 40 |
|  | B2.21 | Tokenizer entraîné sur Holmes, appliqué à Verne (et l'inverse) : le coût de la langue | 🔬 | ★★ | 25 |
| D | B2.22 à B2.27 | vecteurs de mots : co-occurrences et PPMI, cosinus et voisins, PPMI + SVD sur Holmes, word2vec sur Verne, voisins de « Fogg » et « Passepartout », carte des embeddings (*prochaine étape de génération*) | | | |
| E | B2.28 à B2.31 | embeddings de phrases et recherche sémantique, statique contre contextuel, sauvegarder un tokenizer, et le défi (*prochaine étape de génération*) | | | |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")  # your own library

## Objectifs et rappel express

- Manipuler Unicode, UTF-8 et la normalisation d'un texte, et compter ses octets et ses points de code.
- Écrire un tokenizer BPE (entraînement, encodage par rang, décodage) et le comparer à la bibliothèque `tokenizers`.
- Comparer des tokenizers réels et mesurer leur coût (fertilité, compression) selon la langue.
- Relier `nn.Embedding` à un produit one-hot × table, et entraîner des embeddings word2vec.
- Construire des embeddings par co-occurrences (PPMI et SVD), les interroger par le cosinus, et chercher des passages par des embeddings de phrases.

**Rappel express.** Fertilité $= \frac{\text{tokens}}{\text{mots}}$, compression $= \frac{\text{octets}}{\text{tokens}}$. Fusion BPE : la paire voisine la plus fréquente (à égalité, la plus petite paire d'identifiants) devient un token ; on encode en appliquant les fusions par rang. $\mathbf{x}_{\text{one-hot}}^\top \mathbf{E} = \mathbf{E}[i, :]$. $\operatorname{PPMI}(w, c) = \max\big(0, \log \frac{p(w, c)}{p(w)\,p(c)}\big)$. Skip-gram : $L = -\log \sigma(\mathbf{u}_c^\top \mathbf{v}_w) - \sum_k \log \sigma(-\mathbf{u}_{n_k}^\top \mathbf{v}_w)$. Mean pooling : $\mathbf{s} = \frac{\sum_t m_t \mathbf{h}_t}{\sum_t m_t}$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ B2.1 à B2.6, 🧮 B2.9)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"B"`) ; une forme de tenseur : entre guillemets (`"(2, 3)"`) ; un mot : entre guillemets (`"mot"`) ; des octets : une liste, en hexadécimal ou en décimal (`[0x4E, 0x6F]` ou `[78, 111]`) ; des identifiants, une paire d'identifiants ou un vecteur : une liste (`[4, 0, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ B2.7, l'estimation 🧮 B2.8, la réflexion (🗣️ B2.10, ⚖️ B2.11, 📄 B2.12) et l'entretien se corrigent avec `05_solutions.md`.

**B2.Q1 — Caractères, mots ou sous-mots : taille du vocabulaire contre longueur des séquences**

In [ ]:
# B2.Q1: the values you found on paper
answer_B2_Q1a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q1b = ...  # b) True or False
answer_B2_Q1c = ...  # c) a whole number
answer_B2_Q1d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q1a, answer_B2_Q1b, answer_B2_Q1c, answer_B2_Q1d, answer_B2_Q1e]):
    wb.check(f"B2.Q1{letter}", answer)

**B2.Q2 — Octets, points de code, caractères affichés : combien « é » compte-t-il ?**

In [ ]:
# B2.Q2: the values you found on paper
answer_B2_Q2a = ...  # a) a whole number
answer_B2_Q2b = ...  # b) a whole number
answer_B2_Q2c = ...  # c) a whole number
answer_B2_Q2d = ...  # d) a whole number
answer_B2_Q2e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q2a, answer_B2_Q2b, answer_B2_Q2c, answer_B2_Q2d, answer_B2_Q2e]):
    wb.check(f"B2.Q2{letter}", answer)

**B2.Q3 — NFC ou NFD : deux chaînes identiques à l'écran, différentes en mémoire**

In [ ]:
# B2.Q3: the values you found on paper
answer_B2_Q3a = ...  # a) True or False
answer_B2_Q3b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q3c = ...  # c) True or False
answer_B2_Q3d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q3e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q3a, answer_B2_Q3b, answer_B2_Q3c, answer_B2_Q3d, answer_B2_Q3e]):
    wb.check(f"B2.Q3{letter}", answer)

**B2.Q4 — Fusions BPE : pourquoi la liste ordonnée des fusions est le modèle**

In [ ]:
# B2.Q4: the values you found on paper
answer_B2_Q4a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q4b = ...  # b) True or False
answer_B2_Q4c = ...  # c) a whole number
answer_B2_Q4d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q4a, answer_B2_Q4b, answer_B2_Q4c, answer_B2_Q4d, answer_B2_Q4e]):
    wb.check(f"B2.Q4{letter}", answer)

**B2.Q5 — BPE au niveau octet : jamais de token inconnu, vrai ou faux ?**

In [ ]:
# B2.Q5: the values you found on paper
answer_B2_Q5a = ...  # a) True or False
answer_B2_Q5b = ...  # b) a whole number
answer_B2_Q5c = ...  # c) True or False
answer_B2_Q5d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q5e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q5a, answer_B2_Q5b, answer_B2_Q5c, answer_B2_Q5d, answer_B2_Q5e]):
    wb.check(f"B2.Q5{letter}", answer)

**B2.Q6 — Tokens spéciaux et espace initial : « Ġthe » n'est pas « the »**

In [ ]:
# B2.Q6: the values you found on paper
answer_B2_Q6a = ...  # a) True or False
answer_B2_Q6b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q6c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q6d = ...  # d) True or False
answer_B2_Q6e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_Q6a, answer_B2_Q6b, answer_B2_Q6c, answer_B2_Q6d, answer_B2_Q6e]):
    wb.check(f"B2.Q6{letter}", answer)

**B2.Q7 — WordPiece, Unigram, SentencePiece : quel critère pour quelle méthode ?**

In [ ]:
# B2.Q7: the values you found on paper
answer_B2_Q7a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q7b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q7c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_Q7d = ...  # d) True or False
answer_B2_Q7e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_Q7a, answer_B2_Q7b, answer_B2_Q7c, answer_B2_Q7d, answer_B2_Q7e]):
    wb.check(f"B2.Q7{letter}", answer)

**B2.Q8 — Pourquoi un LLM compte mal les lettres d'un mot et calcule mal avec les chiffres**

In [ ]:
# B2.Q8: the values you found on paper
answer_B2_Q8a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q8b = ...  # b) True or False
answer_B2_Q8c = ...  # c) a whole number
answer_B2_Q8d = ...  # d) a whole number
answer_B2_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q8a, answer_B2_Q8b, answer_B2_Q8c, answer_B2_Q8d, answer_B2_Q8e]):
    wb.check(f"B2.Q8{letter}", answer)

**B2.Q9 — nn.Embedding : une table de consultation entraînable, pas une couche magique**

In [ ]:
# B2.Q9: the values you found on paper
answer_B2_Q9a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_Q9b = ...  # b) a whole number
answer_B2_Q9c = ...  # c) True or False
answer_B2_Q9d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q9e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q9a, answer_B2_Q9b, answer_B2_Q9c, answer_B2_Q9d, answer_B2_Q9e]):
    wb.check(f"B2.Q9{letter}", answer)

**B2.Q10 — L'hypothèse distributionnelle : un mot se définit par ses voisins**

In [ ]:
# B2.Q10: the values you found on paper
answer_B2_Q10a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q10b = ...  # b) True or False
answer_B2_Q10c = ...  # c) a number, 2 decimals
answer_B2_Q10d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q10e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_B2_Q10a, answer_B2_Q10b, answer_B2_Q10c, answer_B2_Q10d, answer_B2_Q10e]):
    wb.check(f"B2.Q10{letter}", answer)

**B2.Q11 — Statique ou contextuel : l'« avocat » du tribunal et celui du marché**

In [ ]:
# B2.Q11: the values you found on paper
answer_B2_Q11a = ...  # a) the letter of your choice, e.g. "B"
answer_B2_Q11b = ...  # b) the letter of your choice, e.g. "B"
answer_B2_Q11c = ...  # c) True or False
answer_B2_Q11d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_Q11e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q11a, answer_B2_Q11b, answer_B2_Q11c, answer_B2_Q11d, answer_B2_Q11e]):
    wb.check(f"B2.Q11{letter}", answer)

**B2.Q12 — Cosinus ou distance euclidienne pour des embeddings normalisés ?**

In [ ]:
# B2.Q12: the values you found on paper
answer_B2_Q12a = ...  # a) a number, 1 decimal
answer_B2_Q12b = ...  # b) True or False
answer_B2_Q12c = ...  # c) a number, 3 decimals
answer_B2_Q12d = ...  # d) True or False
answer_B2_Q12e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_Q12a, answer_B2_Q12b, answer_B2_Q12c, answer_B2_Q12d, answer_B2_Q12e]):
    wb.check(f"B2.Q12{letter}", answer)

**B2.R1 — À quoi sert l'embedding d'image extrait d'un réseau pré-entraîné ?**

In [ ]:
# B2.R1: the values you found on paper
answer_B2_R1a = ...  # a) True or False
answer_B2_R1b = ...  # b) a whole number
answer_B2_R1c = ...  # c) the letter of your choice, e.g. "B"
answer_B2_R1d = ...  # d) True or False
answer_B2_R1e = ...  # e) the letter of your choice, e.g. "B"

for letter, answer in zip("abcde", [answer_B2_R1a, answer_B2_R1b, answer_B2_R1c, answer_B2_R1d, answer_B2_R1e]):
    wb.check(f"B2.R1{letter}", answer)

**B2.R2 — build_vocab, encode, decode et le token inconnu**

In [ ]:
# B2.R2: the values you found on paper
answer_B2_R2a = ...  # a) a whole number
answer_B2_R2b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_R2c = ...  # c) True or False
answer_B2_R2d = ...  # d) the letter of your choice, e.g. "B"
answer_B2_R2e = ...  # e) a whole number

for letter, answer in zip("abcde", [answer_B2_R2a, answer_B2_R2b, answer_B2_R2c, answer_B2_R2d, answer_B2_R2e]):
    wb.check(f"B2.R2{letter}", answer)

**B2.R3 — Entropie, Huffman et ce que l'on compresse vraiment**

In [ ]:
# B2.R3: the values you found on paper
answer_B2_R3a = ...  # a) a number, 2 decimals
answer_B2_R3b = ...  # b) a whole number
answer_B2_R3c = ...  # c) True or False
answer_B2_R3d = ...  # d) True or False
answer_B2_R3e = ...  # e) a number, 1 decimal

for letter, answer in zip("abcde", [answer_B2_R3a, answer_B2_R3b, answer_B2_R3c, answer_B2_R3d, answer_B2_R3e]):
    wb.check(f"B2.R3{letter}", answer)

**Ex B2.1 — « Noël à Paris » en UTF-8 : octets, points de code, longueur**

In [ ]:
# Ex B2.1: the values you found on paper
answer_B2_1a = ...  # a) a whole number
answer_B2_1b = ...  # b) a whole number
answer_B2_1c = ...  # c) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1d = ...  # d) a whole number
answer_B2_1e = ...  # e) a whole number
answer_B2_1f = ...  # f) a list of byte values, e.g. [0x4E, 0x6F]
answer_B2_1g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_1a, answer_B2_1b, answer_B2_1c, answer_B2_1d, answer_B2_1e, answer_B2_1f, answer_B2_1g]):
    wb.check(f"B2.1{letter}", answer)

**Ex B2.2 — Analogie à la main : roi − homme + femme sur des vecteurs de dimension 3**

In [ ]:
# Ex B2.2: the values you found on paper
answer_B2_2a = ...  # a) a number, 2 decimals
answer_B2_2b = ...  # b) a number, 3 decimals
answer_B2_2c = ...  # c) a number, 3 decimals
answer_B2_2d = ...  # d) a word as a string, e.g. "mot"
answer_B2_2e = ...  # e) a word as a string, e.g. "mot"
answer_B2_2f = ...  # f) a word as a string, e.g. "mot"
answer_B2_2g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_2a, answer_B2_2b, answer_B2_2c, answer_B2_2d, answer_B2_2e, answer_B2_2f, answer_B2_2g]):
    wb.check(f"B2.2{letter}", answer)

**Ex B2.3 — Trois fusions BPE à la main sur un mini-corpus**

In [ ]:
# Ex B2.3: the values you found on paper
answer_B2_3a = ...  # a) a whole number
answer_B2_3b = ...  # b) a whole number
answer_B2_3c = ...  # c) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3d = ...  # d) a whole number
answer_B2_3e = ...  # e) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3f = ...  # f) a pair of ids as a list or a tuple, e.g. [2, 7]
answer_B2_3g = ...  # g) a whole number
answer_B2_3h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_3a, answer_B2_3b, answer_B2_3c, answer_B2_3d, answer_B2_3e, answer_B2_3f, answer_B2_3g, answer_B2_3h]):
    wb.check(f"B2.3{letter}", answer)

**Ex B2.4 — Encoder un mot jamais vu avec des fusions déjà apprises**

In [ ]:
# Ex B2.4: the values you found on paper
answer_B2_4a = ...  # a) a list of token ids, e.g. [4, 0, 7]
answer_B2_4b = ...  # b) a list of token ids, e.g. [4, 0, 7]
answer_B2_4c = ...  # c) a list of token ids, e.g. [4, 0, 7]
answer_B2_4d = ...  # d) a list of token ids, e.g. [4, 0, 7]
answer_B2_4e = ...  # e) a list of token ids, e.g. [4, 0, 7]
answer_B2_4f = ...  # f) a whole number
answer_B2_4g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_4a, answer_B2_4b, answer_B2_4c, answer_B2_4d, answer_B2_4e, answer_B2_4f, answer_B2_4g]):
    wb.check(f"B2.4{letter}", answer)

**Ex B2.5 — One-hot fois matrice : nn.Embedding à la main, et quelles lignes reçoivent un gradient**

In [ ]:
# Ex B2.5: the values you found on paper
answer_B2_5a = ...  # a) a shape as a string, e.g. "(2, 3)"
answer_B2_5b = ...  # b) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5c = ...  # c) a number, 1 decimal
answer_B2_5d = ...  # d) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5e = ...  # e) a whole number
answer_B2_5f = ...  # f) a list of numbers, e.g. [0.5, -1, 3]
answer_B2_5g = ...  # g) True or False
answer_B2_5h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_5a, answer_B2_5b, answer_B2_5c, answer_B2_5d, answer_B2_5e, answer_B2_5f, answer_B2_5g, answer_B2_5h]):
    wb.check(f"B2.5{letter}", answer)

**Ex B2.6 — Co-occurrences et PPMI à la main (fenêtre de 1, quatre mots)**

In [ ]:
# Ex B2.6: the values you found on paper
answer_B2_6a = ...  # a) a whole number
answer_B2_6b = ...  # b) a whole number
answer_B2_6c = ...  # c) a number, 3 decimals
answer_B2_6d = ...  # d) a number, 3 decimals
answer_B2_6e = ...  # e) a number, 3 decimals
answer_B2_6f = ...  # f) a number, 3 decimals
answer_B2_6g = ...  # g) True or False
answer_B2_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_B2_6a, answer_B2_6b, answer_B2_6c, answer_B2_6d, answer_B2_6e, answer_B2_6f, answer_B2_6g, answer_B2_6h]):
    wb.check(f"B2.6{letter}", answer)

**Ex B2.9 — Fermi : poids de la table d'embedding de GPT-2 et d'un LLM à 128 000 tokens**

In [ ]:
# Ex B2.9: the values you found on paper
answer_B2_9a = ...  # a) a whole number
answer_B2_9b = ...  # b) a number, 1 decimal
answer_B2_9c = ...  # c) a number, 1 decimal
answer_B2_9d = ...  # d) a whole number
answer_B2_9e = ...  # e) a number, 1 decimal
answer_B2_9f = ...  # f) a whole number
answer_B2_9g = ...  # g) True or False

for letter, answer in zip("abcdefg", [answer_B2_9a, answer_B2_9b, answer_B2_9c, answer_B2_9d, answer_B2_9e, answer_B2_9f, answer_B2_9g]):
    wb.check(f"B2.9{letter}", answer)

## Partie A · Des tokenizers réels : ce que coûte un texte

Avant d'écrire ton propre BPE, tu prends en main trois tokenizers de modèles réels et tu mesures ce que leur coûtent Holmes et Verne : leur nombre de tokens, leur fertilité (tokens par mot) et leur compression (octets par token, fiche §202.6). La cellule suivante charge les outils de la phase « tokenisation » (parties A à C) :
- `EXCERPT_H` et `EXCERPT_V` : les 50 000 caractères `[100000:150000]` de chaque roman, figés ;
- `HOLMES_TRAIN`, `HOLMES_VAL`, `VERNE_TRAIN`, `VERNE_VAL` : chaque roman coupé à 90 % / 10 %, au début d'une ligne et sans mélange (un tokenizer appris sur l'entraînement ne voit jamais la validation) ;
- `TOKENIZER_REPOS` : le dépôt du Hugging Face Hub et la révision (un commit fixé) de six tokenizers (`data/cards/pretrained.md`) ; seuls leurs fichiers de tokenizer se téléchargent, quelques Mo ;
- `n_bytes`, `reference_char_bpe` (le BPE de la bibliothèque `tokenizers`, pour les vérifications), `truncated_bpe` (partie C).

**Données** : *The Adventures of Sherlock Holmes* (A. Conan Doyle, Project Gutenberg n° 1661, `data/cards/holmes.md`) et *Le Tour du monde en quatre-vingts jours* (J. Verne, n° 800, `data/cards/verne.md`), dans le domaine public ; tu les as déjà découpés en caractères et en mots au ch. 22. Sans connexion au Hub, les cellules qui ont besoin d'un tokenizer réel affichent ⏳ « à valider sur Colab » et le reste du notebook continue.

In [ ]:
# Tools for the notebook exercises of the tokenization phase (parts A to C)
import json
import math
import os
import re
import time
import unicodedata
import warnings
from collections import Counter
from types import MappingProxyType

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wb import text as wbtext
from wb.nbtools import StopCheck, checked, filled, fr, refuses, returned, run_pytest_group, same, verdict, written

warnings.filterwarnings("ignore", message=".*IProgress not found.*")    # tqdm without ipywidgets (transformers)
try:
    from transformers.utils import logging as _hf_logging
    _hf_logging.set_verbosity_error()     # no « Token indices sequence length is longer… »: we count, no model reads
except ImportError:                       # transformers absent: the cells that need it say so
    pass

TEST_TOK = "tests/test_chb2_tokenizer.py"     # the tests of your library tokenizer.py
EXCERPT_H = wbtext.excerpt("holmes")           # 50 000 characters of Holmes, [100000:150000] (frozen)
EXCERPT_V = wbtext.excerpt("verne")            # 50 000 characters of Verne, [100000:150000]
HOLMES_TRAIN, HOLMES_VAL = wbtext.train_val("holmes")   # 90 % / 10 %, cut at a line, never shuffled
VERNE_TRAIN, VERNE_VAL = wbtext.train_val("verne")
TOKENIZER_REPOS = {name: (spec["repo"], spec["revision"]) for name, spec in wbtext.TOKENIZERS.items()}


def n_bytes(text):
    """The number of UTF-8 bytes of a text."""
    return len(text.encode("utf-8"))


def need(ex_id, *names):
    """Stop a check cell with ⏳ while a function of an earlier exercise is not defined yet."""
    for name in names:
        if globals().get(name) is None:
            print(f"⏳ Ex {ex_id} : définis d'abord {name}.")
            raise StopCheck


def bpe_written():
    """True once BPETokenizer.fit and encode of your library run (B2.16 and B2.17), else False."""
    return written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258, level="byte").fit("aa aa").encode("aa"))


def reference_char_bpe(text, vocab_size, min_frequency=2):
    """The BPE of the library tokenizers trained like BPETokenizer(level="char", pre_tokenizer="whitespace") (fiche
    §202.3): models.BPE, pre_tokenizers.Whitespace, BpeTrainer. For the checks."""
    from tokenizers import Tokenizer, models, pre_tokenizers, trainers
    hf = Tokenizer(models.BPE())
    hf.pre_tokenizer = pre_tokenizers.Whitespace()
    hf.train_from_iterator([text], trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=min_frequency,
                                                       show_progress=False))
    return hf


def truncated_bpe(tok, vocab_size):
    """A copy of a fitted BPETokenizer without special token that keeps only its first vocab_size - n_base_ merges:
    the tokenizer that fit would have learned with this vocab_size (merges are learned one after the other, the same
    first ones whatever the final size, as long as min_frequency does not stop the training)."""
    if tok.special_ids_:
        raise ValueError("truncated_bpe: only for a tokenizer without special token")
    k = vocab_size - tok.n_base_
    if not 0 <= k <= len(tok.merges_):
        raise ValueError(f"truncated_bpe: vocab_size must be between {tok.n_base_} and "
                         f"{tok.n_base_ + len(tok.merges_)}, got {vocab_size}")
    small = type(tok)(vocab_size=vocab_size, level=tok.level, pre_tokenizer=tok.pre_tokenizer,
                      min_frequency=tok.min_frequency)
    small.n_base_, small.merges_, small.special_ids_ = tok.n_base_, list(tok.merges_[:k]), {}
    small.vocab_ = {i: tok.vocab_[i] for i in range(tok.n_base_ + k)}
    return small


print(f"Holmes : {len(HOLMES_TRAIN):_} caractères d'entraînement, {len(HOLMES_VAL):_} de validation ; Verne : "
      f"{len(VERNE_TRAIN):_} et {len(VERNE_VAL):_} ; extraits de {len(EXCERPT_H):_} caractères.".replace("_", " "))

### Ex B2.13 — Tokenizers réels : GPT-2, BERT et un tokenizer multilingue sur Holmes et Verne 📦 ★★ ⏱️ 20 min
**Objectif :** charger trois tokenizers réels à une révision fixée, et mesurer ce que leur coûte un texte anglais et un texte français.  
**Prérequis :** ch. 22 (vocabulaire et encodage) · fiche §202.1, §202.5 (« Utiliser un tokenizer avec `transformers` »), §202.6 (« Deux rapports ») · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

**Contrat**
- `load_tokenizers_13()` : un dictionnaire de trois tokenizers de `transformers`, chacun chargé par `AutoTokenizer.from_pretrained(dépôt, revision=révision)` avec le dépôt et la révision de `TOKENIZER_REPOS` :
  - `"gpt2"`, le BPE octet de GPT-2 ;
  - `"bert"`, le WordPiece de `bert-base-uncased` ;
  - `"e5"`, le tokenizer multilingue de `multilingual-e5-small` (celui de XLM-R).
- `measure_13(tok, text)` : un dictionnaire de quatre mesures :
  - `"tokens"` : le nombre de tokens du texte **seul**, sans les tokens spéciaux que certains tokenizers ajoutent autour d'un texte ;
  - `"words"` : le nombre de mots, `len(text.split())` ;
  - `"fertility"` : tokens / mots ;
  - `"bytes_per_token"` : octets UTF-8 du texte / tokens, la compression.

La cellule de vérification compare tes trois tokenizers à ceux de référence (`wbtext.load_tokenizer`) : leur type, leur taille et les identifiants d'une phrase d'essai. Elle mesure ensuite chacun sur `EXCERPT_H` et `EXCERPT_V` et affiche le tableau des six lignes. Quatre valeurs sont vérifiées, calculées par ta fonction (rien à taper) :
- a) le nombre de tokens de GPT-2 sur l'extrait de Holmes ;
- b) celui de BERT sur l'extrait de Verne ;
- c) la fertilité de e5 sur Holmes ;
- d) la compression de e5 sur Verne.

Dans tes notes : quel tokenizer est le plus économe sur Holmes, et sur Verne ? BERT *uncased* paraît économe sur Verne : `bert.decode(bert(phrase, add_special_tokens=False)["input_ids"])` redonne-t-il une phrase accentuée de Verne ? Que perd-il ? Affiche les tokens d'une même phrase avec les trois tokenizers (`tok.tokenize(phrase)`) : que signalent « Ġ », « ## » et « ▁ » ?

In [ ]:
from transformers import AutoTokenizer


def load_tokenizers_13():
    """{"gpt2": ..., "bert": ..., "e5": ...}: the AutoTokenizer of each repository of TOKENIZER_REPOS, at its revision."""
    raise NotImplementedError("load_tokenizers_13")


def measure_13(tok, text):
    """{"tokens": int, "words": int, "fertility": float, "bytes_per_token": float} for one tokenizer and one text."""
    raise NotImplementedError("measure_13")

In [ ]:
REF_13 = {name: wbtext.try_tokenizer(name, "B2.13") for name in ("gpt2", "bert", "e5")}
KEYS_13 = ("tokens", "words", "fertility", "bytes_per_token")
rows_13 = {}
if any(tok is None for tok in REF_13.values()):
    print("⏳ Ex B2.13 : sans les tokenizers de référence, la vérification attend une connexion au Hub (ou Colab).")
else:
    with checked("B2.13"):
        toks_13 = load_tokenizers_13()
        if not isinstance(toks_13, dict) or set(toks_13) != set(REF_13):
            got = sorted(toks_13) if isinstance(toks_13, dict) else type(toks_13).__name__
            print(f"❌ Ex B2.13 : load_tokenizers_13 doit rendre un dictionnaire de clés 'gpt2', 'bert' et 'e5' ; reçu {got}.")
            raise StopCheck
        probe_13 = "Phileas Fogg paria 20 000 livres, déjà ! 🙂"
        for name, ref in REF_13.items():
            tok = toks_13[name]
            ok = (callable(tok) and type(tok).__name__ == type(ref).__name__ and len(tok) == len(ref)
                  and tok(probe_13)["input_ids"] == ref(probe_13)["input_ids"])
            verdict("B2.13", ok, f"{name} : {type(ref).__name__}, {len(ref):_} tokens, mêmes ids que la référence.".replace("_", " "),
                    f"{name} : ce n'est pas le tokenizer de {TOKENIZER_REPOS[name][0]} (type, taille du vocabulaire ou ids "
                    "d'une phrase d'essai différents de la référence).")
        for name in REF_13:
            for label, text in (("Holmes", EXCERPT_H), ("Verne", EXCERPT_V)):
                m = measure_13(toks_13[name], text)
                if not returned("B2.13", "measure_13", m):
                    raise StopCheck
                if not isinstance(m, dict) or any(key not in m for key in KEYS_13):
                    print(f"❌ Ex B2.13 : measure_13 doit rendre un dictionnaire de clés {', '.join(KEYS_13)} ; reçu "
                          f"{sorted(m) if isinstance(m, dict) else type(m).__name__}.")
                    raise StopCheck
                rows_13[(name, label)] = m
        table_13 = pd.DataFrame({f"{name} · {label}": m for (name, label), m in rows_13.items()}).T
        print(table_13.to_string(float_format=lambda x: f"{x:.3f}"))
        m = rows_13[("gpt2", "Holmes")]
        verdict("B2.13", m["words"] == len(EXCERPT_H.split()) and math.isclose(m["fertility"], m["tokens"] / m["words"])
                and math.isclose(m["bytes_per_token"], n_bytes(EXCERPT_H) / m["tokens"]),
                "les quatre mesures sont cohérentes (mots de split(), fertilité = tokens / mots, octets UTF-8 / tokens).",
                "relis le contrat : words = len(text.split()), fertility = tokens / words, bytes_per_token = octets UTF-8 "
                "du texte / tokens.")
        wb.check("B2.13a", rows_13[("gpt2", "Holmes")]["tokens"], computed=True)
        wb.check("B2.13b", rows_13[("bert", "Verne")]["tokens"], computed=True)
        wb.check("B2.13c", rows_13[("e5", "Holmes")]["fertility"], computed=True)
        wb.check("B2.13d", rows_13[("e5", "Verne")]["bytes_per_token"], computed=True)

### Ex B2.14 — Prédire : quel texte coûte le plus de tokens, Holmes, Verne, des nombres ou des emoji ? 🔮 ★ ⏱️ 10 min
**Objectif :** prédire, en ordres de grandeur, ce que coûtent en tokens un texte anglais, un texte français, des nombres et des emoji.  
**Prérequis :** Ex B2.13 · fiche §202.2 (octets UTF-8), §202.4, §202.5 (WordPiece), §202.6 (« Les chiffres et les lettres ») · **Fil rouge :** Holmes/Verne · **Parcours :** R, C

La cellule suivante prépare deux textes de plus, à côté des deux extraits de B2.13 :
- `NUMBERS_14` : 2 000 nombres de 1 à 5 chiffres, séparés par des espaces (11 768 octets) ;
- `EMOJI_14` : 1 200 emoji collés, sans espace (6 000 octets ; « 👍🏽 » et « ❤️ » font deux points de code chacun).

L'expérience compte les tokens des quatre textes, tokens spéciaux exclus, avec six tokenizers : GPT-2, BERT, e5, CamemBERT, Qwen3 et SmolLM2. **Avant d'exécuter**, écris tes trois prédictions :
- a) `costliest_14` : avec GPT-2, le texte qui a le **moins** d'octets par token (celui qui coûte le plus cher pour sa taille) : `"A"` Holmes, `"B"` Verne, `"C"` les nombres, `"D"` les emoji ;
- b) `ratio_14` : avec GPT-2, le rapport entre les tokens de `EXCERPT_V` et ceux de `EXCERPT_H`, deux textes de 50 000 caractères : `"A"` moins de 1,2, `"B"` de 1,2 à 1,7, `"C"` plus de 1,7 ;
- c) `bert_emoji_14` : le nombre de tokens de BERT (`bert-base-uncased`) pour `EMOJI_14` : `"A"` de 1 à 10, `"B"` de 11 à 2 000, `"C"` plus de 2 000.

La cellule d'expérience n'affiche rien tant que tes trois prédictions ne sont pas écrites ; elle les vérifie à la fin.

In [ ]:
# Two more texts of about 6 000 to 12 000 bytes, next to the two excerpts of B2.13
NUMBERS_14 = " ".join(str(i * 7919 % 100000) for i in range(2000))    # 2 000 numbers of 1 to 5 digits, spaced
EMOJI_14 = "🙂🚀🎉👍🏽🐍❤️" * 200                                        # 1 200 emoji in a row, no space
TEXTS_14 = {"Holmes": EXCERPT_H, "Verne": EXCERPT_V, "nombres": NUMBERS_14, "emoji": EMOJI_14}
print({name: f"{len(text):_} points de code, {n_bytes(text):_} octets".replace("_", " ") for name, text in TEXTS_14.items()})

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
costliest_14 = ...    # a) with GPT-2, the text with the FEWEST bytes per token:
                      #    "A" Holmes, "B" Verne, "C" the numbers, "D" the emoji
ratio_14 = ...        # b) with GPT-2, tokens of EXCERPT_V / tokens of EXCERPT_H (the same 50 000 characters):
                      #    "A" under 1.2, "B" from 1.2 to 1.7, "C" over 1.7
bert_emoji_14 = ...   # c) the number of tokens of BERT (bert-base-uncased) for EMOJI_14, special tokens left out:
                      #    "A" from 1 to 10, "B" from 11 to 2 000, "C" over 2 000

In [ ]:
if any(answer is ... or answer is None for answer in [costliest_14, ratio_14, bert_emoji_14]):
    print("⏳ Ex B2.14 : écris d'abord tes trois prédictions dans la cellule précédente.")
else:
    names_14 = ("gpt2", "bert", "e5", "camembert", "qwen3", "smollm2")
    toks_14 = {name: wbtext.try_tokenizer(name, "B2.14") for name in names_14}
    counts_14 = {name: {label: wbtext.count_tokens(tok, text) for label, text in TEXTS_14.items()}
                 for name, tok in toks_14.items() if tok is not None}
    if "gpt2" not in counts_14 or "bert" not in counts_14:
        print("⏳ Ex B2.14 : l'expérience attend les tokenizers de GPT-2 et de BERT (connexion au Hub, ou Colab).")
    else:
        tokens_14 = pd.DataFrame(counts_14).T
        per_token_14 = pd.DataFrame({name: {label: n_bytes(TEXTS_14[label]) / n for label, n in row.items()}
                                     for name, row in counts_14.items()}).T
        print("Nombre de tokens (tokens spéciaux exclus) :")
        print(tokens_14.to_string())
        print("\nOctets par token :")
        print(per_token_14.to_string(float_format=lambda x: f"{x:.2f}"))
        print(f"\nGPT-2, tokens de Verne / tokens de Holmes : {fr(counts_14['gpt2']['Verne'] / counts_14['gpt2']['Holmes'], 2)}")
        print("BERT sur les emoji :", toks_14["bert"].convert_ids_to_tokens(toks_14["bert"](EMOJI_14, add_special_tokens=False)["input_ids"])[:5])
        for id_14, value_14 in (("B2.14a", costliest_14), ("B2.14b", ratio_14), ("B2.14c", bert_emoji_14)):
            wb.check(id_14, value_14)

Dans tes notes : pour chaque prédiction ratée, qu'est-ce qui t'a trompé ? Compare les six tokenizers texte par texte : lesquels traitent les chiffres un par un (fiche §202.6), et que change ce choix pour les nombres ? Sur quel texte les six tokenizers sont-ils le plus inégaux, et pourquoi ?

## Partie B · Ton BPE : tokenizer.py, puis un tokenizer à réparer et la comparaison à tokenizers

Tu écris dans `mylearn/tokenizer.py` un BPE complet, au niveau caractère et au niveau octet : compter et fusionner les paires (B2.15), apprendre les fusions (B2.16), encoder par rang et décoder sans perte (B2.17). Tu répares ensuite un petit tokenizer qui abîme les accents (B2.18), puis tu compares ton BPE à la bibliothèque `tokenizers`, écrite en Rust : mêmes fusions, autre vitesse (B2.19). Ton `BPETokenizer` resservira en B3 (un mini-GPT sur des tokens BPE).

**Contrat commun de `tokenizer.py`** (les docstrings, plus ces précisions que vérifient les tests) :
- les identifiants : d'abord l'alphabet, `0 … n_base_ − 1` (les 256 octets au niveau octet, ou les caractères des pré-tokens triés par point de code au niveau caractère) ; puis une fusion par identifiant, dans l'ordre d'apprentissage (la fusion de rang $r$ crée l'identifiant `n_base_ + r`) ; puis les tokens spéciaux ;
- tes fonctions ne modifient jamais leurs entrées ; une entrée invalide lève `ValueError`, jamais `KeyError` ni `AttributeError` ;
- la bibliothèque standard seulement ; `pre_tokenize`, `save` et `load` sont fournis, déjà écrits.

### Ex B2.15 — get_pair_counts et merge_pair 🔨 ★★ ⏱️ 20 min
**Objectif :** compter les paires de tokens voisins d'un corpus pondéré, puis fusionner une paire dans une suite d'identifiants.  
**Prérequis :** ✏️ B2.3 · fiche §202.3 (« L'entraînement », « Ta librairie : `tokenizer.py` ») · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat**
- `get_pair_counts(words)` :
  - `words` est un corpus pré-tokenisé, `{mot en tuple d'identifiants: fréquence}` ; tout mapping convient, en lecture seule (les tests en passent un `MappingProxyType` et un `Counter`), et il n'est pas modifié ;
  - rend un `dict` (un `Counter` convient) `{(gauche, droite): compte}` des paires d'identifiants voisins, chaque occurrence comptant la fréquence de son mot ;
  - les paires qui se chevauchent comptent : `(1, 1, 1)` contient deux fois `(1, 1)` ; aucune paire ne franchit la frontière de deux mots ;
  - une fréquence négative lève `ValueError` ; un mot de fréquence 0 n'est pas testé (le plus naturel : il ne produit aucune paire).
- `merge_pair(ids, pair, new_id)` :
  - remplace chaque occurrence de `pair` par `new_id`, de gauche à droite et sans chevauchement (comme `str.replace`) ;
  - rend toujours une **liste neuve** : jamais l'entrée elle-même, même sans aucune occurrence, et jamais un tuple ;
  - accepte un tuple, sans le modifier ; une paire qui n'a pas exactement deux identifiants lève `ValueError`.

La première cellule de vérification essaie tes deux fonctions sur des cas visibles, puis compte les paires des mots de l'extrait de Verne (chaque caractère écrit par son point de code) et les compare à un comptage direct ; elle essaie enfin deux refus. La seconde lance les 42 tests de l'exercice (`-k "test_get_pair_counts_ or test_merge_pair_"`).

Dans tes notes : l'extrait de Verne a 7 948 mots, mais combien de mots distincts (la cellule l'affiche) ? Pourquoi compter les paires sur les mots distincts, avec leur fréquence, plutôt que sur le texte entier ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.15"):
    T = mylearn.tokenizer
    corpus_15 = {(1, 2, 3): 2, (1, 2): 1, (4, 4, 4): 3, (5,): 7}
    arg_15 = dict(corpus_15)                                     # your function receives a copy
    counts_15 = T.get_pair_counts(MappingProxyType(arg_15))      # a read-only view, as in the tests
    if returned("B2.15", "get_pair_counts", counts_15):
        verdict("B2.15", isinstance(counts_15, dict) and dict(counts_15) == {(1, 2): 3, (2, 3): 2, (4, 4): 6},
                "get_pair_counts : (1, 2) 3 fois, (2, 3) 2 fois, (4, 4) 6 fois ; un mot d'un seul id n'a pas de paire.",
                f"get_pair_counts({corpus_15}) doit rendre {{(1, 2): 3, (2, 3): 2, (4, 4): 6}} (chaque paire voisine, "
                f"pondérée par la fréquence du mot, chevauchements compris) ; reçu {counts_15!r}.")
        verdict("B2.15", arg_15 == corpus_15, "le corpus reçu n'a pas changé.", "get_pair_counts a modifié le corpus reçu.")
    words_15 = Counter(tuple(map(ord, w)) for w in EXCERPT_V.split())      # the words of Verne, as code points
    expected_15 = Counter()
    for w, f in words_15.items():
        for p in zip(w, w[1:]):
            expected_15[p] += f
    got_15 = T.get_pair_counts(dict(words_15))
    verdict("B2.15", isinstance(got_15, dict) and dict(got_15) == dict(expected_15),
            f"sur les {len(words_15):_} mots distincts de l'extrait de Verne : {len(expected_15):_} paires, les mêmes comptes "
            "qu'un comptage direct.".replace("_", " "),
            "sur les mots de l'extrait de Verne, tes comptes diffèrent d'un comptage direct : as-tu pondéré par la "
            "fréquence et gardé les paires qui se chevauchent ?")
    refuses("B2.15", "get_pair_counts({(1, 2): -1}) (une fréquence négative)", lambda: T.get_pair_counts({(1, 2): -1}))

    ids_15 = (3, 1, 2, 1, 2, 1, 1, 1)                            # a tuple: merge_pair must accept it
    merged_15 = T.merge_pair(ids_15, (1, 2), 9)
    if returned("B2.15", "merge_pair", merged_15):
        verdict("B2.15", type(merged_15) is list and merged_15 == [3, 9, 9, 1, 1, 1],
                "merge_pair((3, 1, 2, 1, 2, 1, 1, 1), (1, 2), 9) = [3, 9, 9, 1, 1, 1], une list.",
                f"merge_pair({ids_15}, (1, 2), 9) doit rendre la list [3, 9, 9, 1, 1, 1] ; reçu {merged_15!r}.")
    overlap_15 = T.merge_pair([1, 1, 1], (1, 1), 5)
    verdict("B2.15", overlap_15 == [5, 1], "[1, 1, 1] avec (1, 1) donne [5, 1] : sans chevauchement, de gauche à droite.",
            f"merge_pair([1, 1, 1], (1, 1), 5) doit rendre [5, 1] (sans chevauchement, de gauche à droite) ; reçu {overlap_15!r}.")
    same_15 = [4, 5]
    out_15 = T.merge_pair(same_15, (1, 2), 9)
    verdict("B2.15", out_15 == [4, 5] and out_15 is not same_15 and same_15 == [4, 5],
            "sans occurrence de la paire : une liste neuve égale à l'entrée, l'entrée intacte.",
            "sans occurrence de la paire, merge_pair doit rendre une liste neuve (pas l'entrée elle-même), égale à "
            f"l'entrée, et laisser l'entrée intacte ; reçu {out_15!r}, entrée {same_15!r}.")
    refuses("B2.15", "merge_pair([1, 2], (1, 2, 3), 9) (une paire de trois ids)", lambda: T.merge_pair([1, 2], (1, 2, 3), 9))

In [ ]:
if written(lambda: mylearn.tokenizer.get_pair_counts({(1, 2): 1})):
    run_pytest_group(TEST_TOK, "test_get_pair_counts_ or test_merge_pair_")   # 42 tests
else:
    print("⏳ Ex B2.15 : les tests attendent tes fonctions.")

### Ex B2.16 — BPETokenizer.fit : apprendre les fusions, au niveau caractère puis octet 🔨 ★★★ ⏱️ 50 min
**Objectif :** apprendre les fusions d'un BPE au niveau caractère et au niveau octet, avec le départage et les conditions d'arrêt de `tokenizers`.  
**Prérequis :** Ex B2.15 · ✏️ B2.3 · fiche §202.3 (encadré 🕰️ sur le départage), §202.4 (tokens spéciaux) · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat** de `BPETokenizer.fit(texts)` :
- **Entrée et sortie** : il rend `self` et pose les quatre attributs `n_base_`, `merges_`, `vocab_` et `special_ids_`. Une seule `str` est **un** texte ; tout itérable de `str` convient, un générateur aussi, et la liste reçue n'est pas modifiée.
- **Corpus** : chaque texte est découpé par `pre_tokenize(texte, self.pre_tokenizer, tokens spéciaux)`. Les tokens spéciaux, `unk_token` compris, sont **retirés avant tout comptage** : leurs caractères n'entrent pas dans l'alphabet, et les fusions sont apprises sur le reste. Des pré-tokens identiques sont comptés une fois, avec leur fréquence.
- **Alphabet** : au niveau octet, `n_base_ = 256` et `vocab_[i] = bytes([i])`. Au niveau caractère, les caractères distincts des pré-tokens (sans l'espace avec `'whitespace'`), triés par point de code ; `vocab_[i]` contient leurs octets UTF-8.
- **Fusions** : chaque étape fusionne la paire la plus fréquente, et à égalité la plus petite paire `(gauche, droite)`. La fusion de rang $r$ crée l'identifiant `n_base_ + r`, avec `vocab_[n_base_ + r] = vocab_[a] + vocab_[b]`. `merges_` est une liste de **tuples** `(a, b)`, dans l'ordre d'apprentissage.
- **Arrêt** : l'entraînement s'arrête quand `n_base_ + fusions + spéciaux` atteint `vocab_size`, quand il ne reste aucune paire, ou quand la meilleure paire apparaît moins de `min_frequency` fois (`min_frequency = 0` est permis). `vocab_size` compte **tous** les tokens spéciaux, `unk_token` compris quand il est ajouté : avec `vocab_size = n_base_ + spéciaux`, aucune fusion.
- **Tokens spéciaux** : `special_ids_` les place après les fusions, dans l'ordre donné, puis `unk_token` s'il n'y est pas déjà (une seule fois). `vocab_` contient aussi leurs octets UTF-8, donc `len(vocab_) = n_base_ + fusions + spéciaux`. Un token spécial donné deux fois n'est pas testé.
- **Erreurs** : `ValueError` pour un `level` ou un `pre_tokenizer` inconnu, un `unk_token` au niveau octet, et `vocab_size < n_base_ +` spéciaux (au niveau caractère, une fois le corpus lu).

Les tests comparent tes fusions à celles de `tokenizers` aux deux niveaux, sur des extraits de Holmes et de Verne. Une version qui recompte toutes les paires à chaque fusion les passe en quelques secondes ; sur un roman entier, elle met plusieurs dizaines de secondes, que tu sentiras en B2.19 à B2.21 (l'indice 2 dit comment aller plus vite).

La première cellule de vérification essaie `fit` sur trois petites phrases, puis compare tes 400 tokens de l'extrait de Verne à ceux de `BpeTrainer`, essaie le niveau octet avec deux tokens spéciaux, et trois refus. La seconde lance les 49 tests de l'exercice (`-k "test_pre_tokenize_ or test_bpe_fit_"`, dont les 5 de `pre_tokenize`, fourni).

Dans tes notes : pourquoi l'alphabet est-il trié par point de code, et que deviendrait la comparaison à `tokenizers` s'il suivait l'ordre d'apparition des caractères ? Pourquoi `vocab_size` compte-t-il les tokens spéciaux (pense à la table d'embedding d'un modèle, §202.7) ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.16"):
    T = mylearn.tokenizer
    texts_16 = ["le chat dort, le rat court", "les chats et les rats", "un chaton"]
    copy_16 = list(texts_16)
    tok_16 = T.BPETokenizer(vocab_size=24, level="char", min_frequency=1)
    out_16 = tok_16.fit(copy_16)
    verdict("B2.16", out_16 is tok_16, "fit rend self.", "fit doit rendre self (return self à la fin).")
    verdict("B2.16", copy_16 == texts_16, "la liste des textes n'a pas changé.", "fit a modifié la liste des textes reçue.")
    alphabet_16 = sorted(set("".join(texts_16)) - {" "})
    got_alphabet_16 = [tok_16.vocab_[i] for i in range(getattr(tok_16, "n_base_", 0))]
    verdict("B2.16", got_alphabet_16 == [c.encode() for c in alphabet_16],
            f"alphabet : les {len(alphabet_16)} caractères des pré-tokens, triés par point de code (« , » en premier).",
            f"l'alphabet doit être les caractères des pré-tokens (sans les espaces), triés par point de code : attendu "
            f"{''.join(alphabet_16)!r}, reçu {b''.join(got_alphabet_16)!r}.")
    merges_16 = getattr(tok_16, "merges_", None)
    verdict("B2.16", isinstance(merges_16, list) and all(type(m) is tuple for m in merges_16)
            and len(tok_16.vocab_) == tok_16.n_base_ + len(merges_16) + len(tok_16.special_ids_) <= 24,
            f"merges_ : une liste de {len(merges_16 or [])} tuples ; len(vocab_) = n_base_ + fusions + spéciaux ≤ vocab_size.",
            "merges_ doit être une liste de tuples (a, b), et vocab_ contenir l'alphabet, les fusions et les tokens "
            "spéciaux, vocab_size au plus.")
    # the same merges as BpeTrainer of tokenizers, on the Verne excerpt
    hf_16 = reference_char_bpe(EXCERPT_V, 400)
    mine_16 = T.BPETokenizer(vocab_size=400, level="char").fit(EXCERPT_V)
    want_16 = [hf_16.id_to_token(i) for i in range(hf_16.get_vocab_size())]
    got_16 = [mine_16.vocab_[i].decode("utf-8", errors="replace") for i in range(len(mine_16.vocab_))]
    first_16 = next((i for i, (a, b) in enumerate(zip(got_16, want_16)) if a != b), None)
    verdict("B2.16", got_16 == want_16,
            f"extrait de Verne, niveau caractère, 400 tokens : les mêmes {len(want_16)} tokens, dans le même ordre, que "
            "BpeTrainer de tokenizers.",
            (f"extrait de Verne, 400 tokens : tes tokens diffèrent de ceux de BpeTrainer dès l'id {first_16} (toi "
             f"{got_16[first_16]!r}, tokenizers {want_16[first_16]!r}) : relis le départage et le tri de l'alphabet."
             if first_16 is not None else f"extrait de Verne : {len(got_16)} tokens chez toi, {len(want_16)} chez "
             "BpeTrainer : relis les conditions d'arrêt (vocab_size, min_frequency)."))
    # byte level, special tokens
    byte_16 = T.BPETokenizer(vocab_size=256 + 10 + 2, pre_tokenizer="gpt2", special_tokens=["<|endoftext|>", "<pad>"])
    byte_16.fit(EXCERPT_V)
    verdict("B2.16", byte_16.n_base_ == 256 and all(byte_16.vocab_[i] == bytes([i]) for i in range(256))
            and len(byte_16.merges_) == 10 and byte_16.special_ids_ == {"<|endoftext|>": 266, "<pad>": 267},
            "niveau octet : 256 octets, 10 fusions, puis <|endoftext|> en 266 et <pad> en 267 (vocab_size compte les spéciaux).",
            "niveau octet avec vocab_size = 268 et deux tokens spéciaux : attendu n_base_ = 256 (vocab_[i] = bytes([i])), "
            f"10 fusions, special_ids_ = {{'<|endoftext|>': 266, '<pad>': 267}} ; reçu n_base_ = "
            f"{getattr(byte_16, 'n_base_', None)}, {len(getattr(byte_16, 'merges_', []))} fusions, "
            f"special_ids_ = {getattr(byte_16, 'special_ids_', None)}.")
    refuses("B2.16", "level='word'", lambda: T.BPETokenizer(level="word").fit("abc"))
    refuses("B2.16", "unk_token au niveau octet", lambda: T.BPETokenizer(unk_token="<unk>").fit("abc"))
    refuses("B2.16", "vocab_size = 100 au niveau octet (moins que les 256 octets)", lambda: T.BPETokenizer(vocab_size=100).fit("abc"))

In [ ]:
if written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258).fit("aa aa")):
    run_pytest_group(TEST_TOK, "test_pre_tokenize_ or test_bpe_fit_")   # 49 tests
else:
    print("⏳ Ex B2.16 : les tests attendent ta méthode fit.")

### Ex B2.17 — encode et decode : fusions appliquées par rang, aller-retour sans perte 🔨 ★★★ ⏱️ 40 min
**Objectif :** encoder un texte en appliquant les fusions par rang, puis le décoder sans perte au niveau octet.  
**Prérequis :** Ex B2.16 · ✏️ B2.4 · fiche §202.3 (« Encoder un mot nouveau : les fusions par rang »), §202.4 · **Fil rouge :** Holmes/Verne · **mylearn :** `tokenizer.py` · **Parcours :** R, M, C

> **Mode d'emploi mylearn** : ouvre `mon_travail/mylearn/tokenizer.py` (créé par `python tools/start_chapter.py B2`), lis la docstring, remplace chaque `raise NotImplementedError(...)` de l'exercice par ton code et **enregistre**. La bibliothèque standard seulement. La première cellule de vérification recharge ta librairie et l'essaie sur des cas visibles ; la seconde lance les tests de l'exercice. `python -m pytest tests/test_chb2_tokenizer.py -q` les lance tous dans un terminal.

**Contrat**
- `encode(text)` rend une `list` d'`int` (`[]` pour un texte vide) :
  - toute clé de `special_ids_` écrite dans le texte devient son identifiant, `unk_token` compris : « <unk> » tapé tel quel donne l'identifiant de `<unk>` ;
  - dans chaque autre pré-token, jamais entre deux, on part des identifiants de base (octets UTF-8, ou caractères), puis on fusionne la paire voisine de **plus petit rang** parmi celles de `merges_`, et ainsi de suite jusqu'à ce qu'aucune paire voisine ne soit une fusion apprise ;
  - au niveau caractère, chaque caractère inconnu donne l'identifiant de `unk_token`, un par caractère.
- `decode(ids)` recolle les octets de tous les tokens, **puis** décode l'UTF-8 avec `errors="replace"` : `[0xC3, 0xA9]` donne « é », et `[0xC3]` seul donne « � ». Il accepte tout entier, Python ou NumPy, qui est une clé de `vocab_`. Avec `'whitespace'`, les espaces disparaissent.
- `tokenize(text)` rend les tokens de `encode(text)` en `str`, « � » pour un morceau de caractère.
- `ValueError`, jamais `AttributeError` ni `KeyError`, pour :
  - un tokenizer non entraîné, dans `encode` comme dans `decode` ;
  - un caractère inconnu sans `unk_token` ;
  - un identifiant inconnu passé à `decode`, −1 compris (ce n'est pas un indice Python).
- `encode` et `decode` ne lisent que les hyperparamètres et les quatre attributs appris : un cache privé posé par `fit` casse le tokenizer rechargé par `load` (testé).

Les tests comparent tes identifiants à ceux de `tokenizers`, vérifient l'aller-retour sans perte sur environ 300 chaînes Unicode tirées au hasard (emoji, idéogrammes, accents combinants, espaces multiples), et que ton nombre de tokens reste à moins de 2 % de celui d'un BPE `ByteLevel` de même taille.

La première cellule de vérification compare ton encodage d'un passage de Verne à celui de `tokenizers`, essaie `<unk>`, l'aller-retour d'un texte accentué avec un emoji et des idéogrammes, un octet seul, `tokenize`, et quatre refus. La seconde lance les 36 tests de l'exercice (`-k "test_bpe_encode_ or test_bpe_decode_ or test_bpe_tokenize_ or test_bpe_save_load_"`).

Dans tes notes : avec `'whitespace'`, que redonne `decode(encode("le chat dort"))`, et pourquoi le pré-tokenizer `'gpt2'` règle-t-il ce problème ? Que rend `tokenize("été")` au niveau octet avec seulement 256 tokens, et pourquoi ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="B2")   # reload your saved file
with checked("B2.17"):
    T = mylearn.tokenizer
    char_17 = T.BPETokenizer(vocab_size=400, level="char").fit(EXCERPT_V)
    hf_17 = reference_char_bpe(EXCERPT_V, 400)
    sample_17 = EXCERPT_V[20000:24000]                    # characters seen during fit
    ids_17 = char_17.encode(sample_17)
    if returned("B2.17", "encode", ids_17):
        verdict("B2.17", type(ids_17) is list and all(isinstance(i, int) for i in ids_17) and ids_17 == hf_17.encode(sample_17).ids,
                f"niveau caractère : les {len(ids_17)} ids d'un passage de Verne sont ceux de tokenizers (fusions par rang).",
                "niveau caractère : encode doit rendre une list d'int identique aux ids de tokenizers (BpeTrainer) ; "
                "applique-t-il les fusions par rang, dans chaque pré-token ?")
    verdict("B2.17", char_17.encode("") == [], "encode('') = [].", f"encode('') doit rendre [] ; reçu {char_17.encode('')!r}.")
    unk_17 = T.BPETokenizer(vocab_size=300, level="char", unk_token="<unk>").fit(EXCERPT_V)
    u_17 = unk_17.special_ids_.get("<unk>")
    got_unk_17 = unk_17.encode("Fogg 🙂🙂 <unk>")
    verdict("B2.17", u_17 is not None and got_unk_17[-3:] == [u_17] * 3 and got_unk_17.count(u_17) == 3,
            "niveau caractère : chaque caractère inconnu donne l'id de <unk>, et « <unk> » écrit dans le texte aussi.",
            f"« Fogg 🙂🙂 <unk> » : attendu trois fois l'id {u_17} de <unk> (un par emoji inconnu, un pour « <unk> » "
            f"écrit tel quel) ; reçu {got_unk_17!r}.")
    byte_17 = T.BPETokenizer(vocab_size=600, pre_tokenizer="gpt2").fit(EXCERPT_V)
    text_17 = VERNE_VAL[:3000] + " 🙂 漢字, é !"
    back_17 = byte_17.decode(byte_17.encode(text_17))
    verdict("B2.17", back_17 == text_17,
            "niveau octet, pré-tokenizer 'gpt2' : decode(encode(texte)) redonne le texte exact (emoji et idéogrammes compris).",
            "niveau octet avec 'gpt2', decode(encode(texte)) doit redonner le texte exact ; il diffère dès le caractère "
            f"{next((i for i, (a, b) in enumerate(zip(back_17, text_17)) if a != b), min(len(back_17), len(text_17)))}.")
    lone_17 = byte_17.decode([0xC3])
    verdict("B2.17", lone_17 == "�", "decode([0xC3]) (un octet seul) = '�'.",
            f"decode([0xC3]), le premier octet de « é » seul, doit rendre '�' ; reçu {lone_17!r}.")
    pieces_17 = byte_17.tokenize("été")
    verdict("B2.17", isinstance(pieces_17, list) and all(isinstance(p, str) for p in pieces_17) and "".join(pieces_17) == "été",
            f"tokenize('été') = {pieces_17!r}.", f"tokenize doit rendre une liste de str ; reçu {pieces_17!r}.")
    refuses("B2.17", "encode avant fit", lambda: T.BPETokenizer().encode("abc"))
    refuses("B2.17", "decode([-1])", lambda: byte_17.decode([-1]))
    refuses("B2.17", "decode([len(vocab_)])", lambda: byte_17.decode([len(byte_17.vocab_)]))
    refuses("B2.17", "niveau caractère sans unk_token : encode('🙂')", lambda: char_17.encode("🙂"))

In [ ]:
if written(lambda: mylearn.tokenizer.BPETokenizer(vocab_size=258).fit("aa aa").encode("aa")):
    run_pytest_group(TEST_TOK, "test_bpe_encode_ or test_bpe_decode_ or test_bpe_tokenize_ or test_bpe_save_load_")   # 36 tests
else:
    print("⏳ Ex B2.17 : les tests attendent tes méthodes fit et encode.")

### Ex B2.18 — Le tokenizer qui abîme les accents : trois bugs d'Unicode et de fusions 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver et corriger les trois erreurs d'un petit tokenizer BPE, guidé par des diagnostics enchaînés.  
**Prérequis :** Ex B2.17 · fiche §202.2 (normalisation), §202.3, §202.4 · tableau des pièges ⚠️ · **Fil rouge :** Verne · **Parcours :** R, C

Un collègue a écrit un petit tokenizer BPE au niveau octet, avec les pré-tokens de GPT-2, en cinq fonctions : `clean_18`, `merge_18`, `train_18`, `encode_18` et `decode_18`. Il tourne, mais il abîme le français. Le code contient **trois** erreurs. Corrige-le **sur place**, dans sa cellule, sans changer les noms, les signatures ni ce que rendent les fonctions.

Ce qu'il devrait faire, comme ton `BPETokenizer` de B2.17 :
- `train_18(text, vocab_size)` rend `(merges, words)` : les fusions dans l'ordre d'apprentissage, et chaque pré-token du texte avec ses identifiants à la fin de l'entraînement ;
- `encode_18(text, merges)` encode par rang ;
- `decode_18(ids, merges)` redonne le texte exact, et un octet seul devient « � ».

La cellule de vérification lance `diagnose_18` : un entraînement de 300 tokens sur les 120 000 premiers caractères de `VERNE_TRAIN` (environ 1 s), puis trois diagnostics enchaînés, chacun ne s'exécutant que si le précédent passe, car une erreur peut en cacher une autre :
- `decode` redonne-t-il un texte d'essai ?
- comment « é » est-il encodé ?
- `encode` découpe-t-il les mots du texte d'entraînement comme l'entraînement les a laissés ?

Elle tourne dès le premier « Tout exécuter » : ses messages sont ton point de départ. Une fois les trois points passés, la cellule suivante mesure ta version réparée sur la validation de Verne.

Dans tes notes : les trois erreurs, et pour chacune le symptôme qu'elle produit seule. Laquelle aurait laissé passer un test d'aller-retour sur un texte anglais sans accent ni emoji ? Laquelle ne change jamais le texte rendu par `decode`, seulement le nombre de tokens ?

In [ ]:
TRAIN_18 = VERNE_TRAIN[:120_000]          # the training text of the diagnostics (≈ 1 s per training)
PROBE_18 = "« Phileas Fogg a gagné son pari ! » 🙂"


def diagnose_18(train, encode, decode, vocab_size=300):
    """Three diagnoses of a small tokenizer, in a row: the first one that fails stops the chain (one error can hide
    another). Returns True when the three pass."""
    result = train(TRAIN_18, vocab_size)
    if not (isinstance(result, tuple) and len(result) == 2 and isinstance(result[1], dict)):
        print(f"❌ Ex B2.18 : train_18 doit rendre le couple (merges, words) ; reçu {type(result).__name__}.")
        return False
    merges, words = result
    vocab = {i: bytes([i]) for i in range(256)}
    for r, (a, b) in enumerate(merges):
        vocab[256 + r] = vocab[a] + vocab[b]

    def shown(ids):
        return "[" + ", ".join(repr(vocab[i].decode("utf-8", errors="replace")) if i in vocab else str(i) for i in ids) + "]"

    def point1():
        try:
            back = decode(encode(PROBE_18, merges), merges)
        except UnicodeDecodeError as exc:
            return False, f"decode(encode(texte)) lève UnicodeDecodeError ({exc.reason}) sur « {PROBE_18} »", \
                "ce texte contient des caractères de plusieurs octets : à quel moment decode transforme-t-il des octets en texte ?"
        if back != PROBE_18:
            same_screen = unicodedata.normalize("NFC", back) == PROBE_18
            return False, f"decode(encode(texte)) rend {back!r} au lieu de {PROBE_18!r}", \
                ("les deux textes s'affichent pareil, mais leurs points de code diffèrent : compare-les"
                 if same_screen else "quels caractères changent, et combien d'octets ont-ils ?")
        try:
            lone = decode([0xC3], merges)
        except UnicodeDecodeError as exc:
            return False, f"decode([0xC3]) (le premier octet de « é », seul) lève UnicodeDecodeError ({exc.reason})", \
                "relis la docstring de decode_18 : que doit devenir un octet seul ?"
        if lone != "�":
            return False, f"decode([0xC3]) rend {lone!r} au lieu de '�'", "relis la docstring de decode_18."
        return True, "le texte d'essai revient à l'identique, et un octet seul donne '�'", ""

    def point2():
        ids = encode("é", merges)
        n = TRAIN_18.count("é")
        if len(ids) == 1:
            return True, f"« é » ({n} fois dans le texte d'entraînement) est un seul token", ""
        return False, f"« é » apparaît {n} fois dans le texte d'entraînement, mais encode le découpe en {len(ids)} tokens {[vocab.get(i, i) for i in ids]}", \
            "les fusions apprises pour « é » portent-elles sur les octets que reçoit encode ?"

    def point3():
        counts = Counter(PIECES_18.findall(TRAIN_18))
        diff = sorted((w for w in words if encode(w, merges) != words[w]), key=lambda w: (-counts.get(w, 0), w))
        if not diff:
            return True, f"les {len(words)} mots distincts du texte d'entraînement sont découpés par encode comme à la fin de l'entraînement", ""
        w = diff[0]
        return False, (f"{len(diff)} des {len(words)} mots distincts du texte d'entraînement ne sont pas découpés par encode "
                       f"comme à la fin de l'entraînement ; par exemple {w!r} : {shown(words[w])} à l'entraînement, "
                       f"{shown(encode(w, merges))} par encode"), \
            "compare les deux découpages : laquelle de ses fusions encode a-t-il faite en premier ?"

    points = [("decode redonne le texte", point1), ("« é » est un token", point2),
              ("encode découpe comme l'entraînement", point3)]
    for k, (name, run) in enumerate(points, start=1):
        ok, measured, why = run()
        if ok:
            print(f"✅ Ex B2.18 : point {k}, {name} ({measured}).")
        else:
            print(f"❌ Ex B2.18 : point {k} : {measured} ; {why}")
            for later in range(k + 1, len(points) + 1):
                print(f"⏳ Ex B2.18 : point {later} : corrige d'abord le point {k}.")
            return False
    print("✅ Ex B2.18 : les trois diagnostics passent.")
    return True

In [ ]:
# The tokenizer to repair IN PLACE (same names, same signatures, same returns): the diagnostics of the next cell start
# from it
PIECES_18 = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+")   # GPT-2 pre-tokens


def clean_18(text):
    """The text as the tokenizer learns it: accents written in a single standard form."""
    return unicodedata.normalize("NFD", text)


def merge_18(ids, pair, new_id):
    """A new list where each occurrence of pair, left to right and without overlap, becomes new_id."""
    out, i = [], 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i + 1]) == pair:
            out.append(new_id)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


def train_18(text, vocab_size):
    """Byte-level BPE learned on text (GPT-2 pre-tokens; ties go to the smallest pair). Returns (merges, words): the
    merges in learning order, and {pre-token: its ids at the end of the training}."""
    text = clean_18(text)
    counts = Counter(PIECES_18.findall(text))
    words = {w: list(w.encode("utf-8")) for w in counts}
    merges = []
    while 256 + len(merges) < vocab_size:
        pairs = Counter()
        for w, ids in words.items():
            for pair in zip(ids, ids[1:]):
                pairs[pair] += counts[w]
        if not pairs:
            break
        best = min(pairs, key=lambda p: (-pairs[p], p))
        merges.append(best)
        words = {w: merge_18(ids, best, 255 + len(merges)) for w, ids in words.items()}
    return merges, words


def encode_18(text, merges):
    """The ids of text: inside each pre-token, its UTF-8 bytes, then the learned merges until none applies."""
    rank = {pair: r for r, pair in enumerate(merges)}
    out = []
    for piece in PIECES_18.findall(text):
        ids = list(piece.encode("utf-8"))
        i = 0
        while i < len(ids) - 1:
            pair = (ids[i], ids[i + 1])
            if pair in rank:
                ids[i:i + 2] = [256 + rank[pair]]
                i = 0
            else:
                i += 1
        out += ids
    return out


def decode_18(ids, merges):
    """The text of ids; a byte left alone (part of a character) becomes '�'."""
    vocab = {i: bytes([i]) for i in range(256)}
    for r, (a, b) in enumerate(merges):
        vocab[256 + r] = vocab[a] + vocab[b]
    return "".join(vocab[i].decode("utf-8") for i in ids)

In [ ]:
with checked("B2.18"):
    ok_18 = diagnose_18(train_18, encode_18, decode_18)

In [ ]:
if not globals().get("ok_18"):
    print("⏳ Ex B2.18 : la comparaison attend que les trois points de la cellule précédente passent.")
else:
    with checked("B2.18"):
        merges_18, _ = train_18(TRAIN_18, 300)
        val_18 = VERNE_VAL[:20_000]
        ids_18 = encode_18(val_18, merges_18)
        print(f"Ta version réparée : {fr(n_bytes(val_18) / len(ids_18), 2)} octets par token sur 20 000 caractères de la "
              f"validation de Verne ; « {PROBE_18} » →", [decode_18([i], merges_18) for i in encode_18(PROBE_18, merges_18)])

### Ex B2.19 — Ton BPE contre la bibliothèque tokenizers : mêmes fusions, autre vitesse 📦 ★★ ⏱️ 25 min
**Objectif :** entraîner un BPE avec la bibliothèque `tokenizers`, lire ses fusions dans sa description JSON, et vérifier que ton BPE apprend exactement les mêmes.  
**Prérequis :** Ex B2.17 · fiche §202.3 (encadré 🕰️ sur le départage), §202.5 (« Les tokenizers des modèles, lus dans leur `tokenizer.json` », « Entraîner son propre tokenizer avec `tokenizers` ») · **Fil rouge :** Holmes/Verne · **Parcours :** M, C

**Contrat**
- `train_hf_19(text, vocab_size, min_frequency=2)` : un `Tokenizer` de `tokenizers`, de modèle `models.BPE()` et de pré-tokenizer `pre_tokenizers.Whitespace()`, entraîné sur `text` par un `BpeTrainer` de ces `vocab_size` et `min_frequency`, sans token spécial ni barre de progression ;
- `hf_merges_19(hf_tok)` : ses fusions, en liste de tuples `(id gauche, id droite)` d'`int`, dans l'ordre d'apprentissage. Lis-les dans sa description JSON, `hf_tok.to_str()`, qui est le contenu de son `tokenizer.json`, avec son vocabulaire `hf_tok.get_vocab()`.

La cellule de vérification entraîne ta fonction sur `VERNE_TRAIN` avec 2 000 tokens. Elle vérifie trois valeurs, calculées par tes fonctions (rien à taper) :
- a) la taille de l'alphabet de base, la taille du vocabulaire moins le nombre de fusions ;
- b) le rang de la fusion qui crée le token « Passepartout » ;
- c) le nombre de tokens de `VERNE_VAL`.

Elle entraîne ensuite **ton** `BPETokenizer` de la même façon (niveau caractère, `'whitespace'`, 2 000 tokens), compare ses fusions et ses identifiants à ceux de `tokenizers`, et affiche les deux temps d'entraînement. Ce temps est une mesure, pas une réponse : il dépend de la machine, et de ta façon de compter les paires (de 1 s à une minute environ).

Dans tes notes : combien de fois ton entraînement est-il plus lent que celui de `tokenizers`, et d'où vient l'écart (le langage, l'algorithme) ? Que dit le rang de « Passepartout » de sa fréquence dans le roman ? Pourquoi la comparaison exacte n'est-elle possible qu'avec la même règle de départage et le même ordre de l'alphabet ?

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers


def train_hf_19(text, vocab_size, min_frequency=2):
    """A Tokenizer of the library tokenizers: models.BPE, pre_tokenizers.Whitespace, trained by BpeTrainer on text
    (no special token, no progress bar)."""
    raise NotImplementedError("train_hf_19")


def hf_merges_19(hf_tok):
    """The merges of a trained BPE of tokenizers, as (left_id, right_id) tuples of ints, in learning order (rank)."""
    raise NotImplementedError("hf_merges_19")

In [ ]:
VOCAB_19 = 2000
with checked("B2.19"):
    start_19 = time.perf_counter()
    hf_19 = train_hf_19(VERNE_TRAIN, VOCAB_19)
    seconds_hf_19 = time.perf_counter() - start_19
    merges_hf_19 = hf_merges_19(hf_19)
    if not returned("B2.19", "hf_merges_19", merges_hf_19):
        raise StopCheck
    if not (isinstance(merges_hf_19, list) and all(type(m) is tuple and len(m) == 2 and all(isinstance(i, int) for i in m)
                                                   for m in merges_hf_19)):
        print("❌ Ex B2.19 : hf_merges_19 doit rendre une liste de tuples de deux int (les ids des deux tokens fusionnés).")
        raise StopCheck
    n_base_19 = hf_19.get_vocab_size() - len(merges_hf_19)
    strings_19 = {i: s for s, i in hf_19.get_vocab().items()}
    rank_19 = next((r for r, (a, b) in enumerate(merges_hf_19) if strings_19.get(a, "") + strings_19.get(b, "") == "Passepartout"), None)
    val_ids_19 = hf_19.encode(VERNE_VAL).ids
    print(f"tokenizers : {hf_19.get_vocab_size()} tokens, dont {len(merges_hf_19)} fusions, appris en {fr(seconds_hf_19, 2)} s ; "
          f"la validation de Verne fait {len(val_ids_19)} tokens.")
    wb.check("B2.19a", n_base_19, computed=True)
    wb.check("B2.19b", rank_19, computed=True)
    wb.check("B2.19c", len(val_ids_19), computed=True)

if not bpe_written():
    print("⏳ Ex B2.19 : la comparaison à ton BPE attend BPETokenizer.fit et encode (B2.16 et B2.17).")
elif globals().get("merges_hf_19") is not None:
    with checked("B2.19"):
        start_19 = time.perf_counter()
        mine_19 = mylearn.tokenizer.BPETokenizer(vocab_size=VOCAB_19, level="char", pre_tokenizer="whitespace").fit(VERNE_TRAIN)
        seconds_mine_19 = time.perf_counter() - start_19
        first_19 = next((r for r, (a, b) in enumerate(zip(mine_19.merges_, merges_hf_19)) if tuple(a) != tuple(b)), None)
        verdict("B2.19", list(map(tuple, mine_19.merges_)) == merges_hf_19,
                f"ton BPE et tokenizers apprennent les mêmes {len(merges_hf_19)} fusions, dans le même ordre.",
                (f"ta fusion de rang {first_19} est {mine_19.merges_[first_19]}, celle de tokenizers {merges_hf_19[first_19]}"
                 if first_19 is not None else f"{len(mine_19.merges_)} fusions chez toi, {len(merges_hf_19)} chez tokenizers")
                + " : relis le départage et l'alphabet (B2.16), puis hf_merges_19.")
        verdict("B2.19", mine_19.encode(VERNE_VAL) == val_ids_19,
                "et les mêmes ids pour toute la validation de Verne.",
                "les ids de la validation diffèrent de ceux de tokenizers : relis encode (B2.17).")
        print(f"Temps de l'entraînement : ton BPETokenizer {fr(seconds_mine_19, 2)} s, tokenizers {fr(seconds_hf_19, 2)} s "
              f"(rapport {seconds_mine_19 / seconds_hf_19:.0f}, sur cette machine, pour cette exécution).")

## Partie C · Expériences : la taille du vocabulaire et la langue d'entraînement

Deux expériences avec ton `BPETokenizer` au niveau octet et le pré-tokenizer `'gpt2'` (sans perte). D'abord la taille du vocabulaire, de 300 à 8 000 tokens (B2.20) ; puis ce que coûte un tokenizer appris sur une langue quand on l'applique à l'autre (B2.21). Les mesures se font toujours sur la **validation**, que les tokenizers n'ont jamais vue. Les fusions s'apprennent une à une, les premières étant les mêmes quelle que soit la taille finale : un seul entraînement à 8 000 tokens suffit, puis `truncated_bpe(tok, taille)` (dans les outils) garde ses `taille − 256` premières fusions. Avec un comptage incrémental des paires, chaque entraînement prend 1 à 2 s ; en recomptant tout à chaque fusion, environ 3 min pour 8 000 tokens et 40 s pour 2 000.

### Ex B2.20 — Taille du vocabulaire : compression et entropie par token, de 300 à 8 000 tokens 🔬 ★★★ ⏱️ 40 min
**Objectif :** mesurer ce qu'apporte un vocabulaire plus grand, en compression et en information par token, pour choisir une taille.  
**Prérequis :** Ex B2.19 · ch. 6 (entropie, code de Huffman) · fiche §202.3, §202.6 · **Fil rouge :** Holmes/Verne · **Parcours :** M, C

Quelle taille de vocabulaire choisir ? Plus de tokens raccourcissent les textes, mais agrandissent la table d'embedding et rendent chaque token plus rare. Tu mesures sur `VERNE_VAL`, pour six tailles de 300 à 8 000 :
- la **compression**, en octets par token ;
- l'**entropie** de la distribution des tokens, en bits par token : $H = -\sum_t p_t \log_2 p_t$, où $p_t$ est la fréquence de l'identifiant $t$ dans le texte encodé (ch. 6) ;
- les **bits par octet**, $H$ divisé par la compression : ce que coûterait chaque octet du texte avec un code de Huffman idéal sur les tokens, si on les supposait indépendants (ch. 6).

**Contrat**
- `compression_20(tok, text)` : le nombre d'octets UTF-8 de `text` divisé par le nombre d'identifiants de `tok.encode(text)` ;
- `token_entropy_20(tok, text)` : l'entropie en bits (logarithme de base 2) des fréquences des identifiants de `tok.encode(text)` ;
- `sweep_20(big_tok, text, sizes)` : une ligne par taille de `sizes`, dans leur ordre ; chaque ligne est un dictionnaire de clés `"vocab_size"`, `"bytes_per_token"`, `"bits_per_token"`, `"bits_per_byte"` et `"used"` (le nombre d'identifiants distincts du texte encodé), mesurées sur `text` avec `truncated_bpe(big_tok, size)`.

La cellule de vérification essaie tes deux mesures sur un faux tokenizer, qui rend des identifiants fixés à l'avance. La cellule d'expérience entraîne ton BPE une fois à 8 000 tokens sur `VERNE_TRAIN` ; elle vérifie à 400 tokens que `truncated_bpe` redonne bien ce qu'aurait appris `fit`, puis lance `sweep_20` et trace les trois courbes.

Dans tes notes :
- la compression gagne-t-elle autant à chaque doublement de la taille ?
- l'entropie par token et les bits par octet : lequel monte, lequel baisse, et comment les deux vont-ils ensemble ?
- combien des 8 000 tokens servent dans la validation (colonne `used`) ?
- quelle taille prendrais-tu pour un petit modèle de langue sur Verne, et que ces courbes ne disent-elles pas (la taille de la table d'embedding, 🧮 B2.9 ; les tokens rares, vus trop peu pour être bien appris) ?

In [ ]:
def compression_20(tok, text):
    """The bytes per token of text: its number of UTF-8 bytes / the number of ids of tok.encode(text)."""
    raise NotImplementedError("compression_20")


def token_entropy_20(tok, text):
    """The entropy, in bits, of the frequencies of the ids of tok.encode(text)."""
    raise NotImplementedError("token_entropy_20")


def sweep_20(big_tok, text, sizes):
    """One dict per size of sizes, in their order: {"vocab_size", "bytes_per_token", "bits_per_token",
    "bits_per_byte", "used"} measured on text with truncated_bpe(big_tok, size); bits_per_byte = bits_per_token /
    bytes_per_token, "used" = the number of distinct ids of the encoded text."""
    raise NotImplementedError("sweep_20")

In [ ]:
class ListTokenizer_20:
    """A fake tokenizer for the check: encode gives the ids it was built with, whatever the text."""
    def __init__(self, ids):
        self.ids = list(ids)

    def encode(self, text):
        return list(self.ids)


with checked("B2.20"):
    c_20 = compression_20(ListTokenizer_20([7, 8]), "été")
    if returned("B2.20", "compression_20", c_20):
        verdict("B2.20", isinstance(c_20, (int, float)) and math.isclose(c_20, 2.5),
                "compression_20 : « été » en 2 tokens, 5 octets UTF-8, soit 2,5 octets par token.",
                f"« été » (5 octets UTF-8, 3 caractères) encodé en 2 tokens : attendu 2,5 octets par token ; reçu {c_20!r}.")
    h_20 = token_entropy_20(ListTokenizer_20([4, 4, 9, 2]), "x")
    if returned("B2.20", "token_entropy_20", h_20):
        verdict("B2.20", isinstance(h_20, (int, float)) and math.isclose(h_20, 1.5),
                "token_entropy_20 : les ids [4, 4, 9, 2] ont une entropie de 1,5 bit.",
                f"les ids [4, 4, 9, 2] (fréquences 1/2, 1/4, 1/4) : attendu 1,5 bit (logarithme de base 2) ; reçu {h_20!r}.")
    verdict("B2.20", math.isclose(token_entropy_20(ListTokenizer_20([3, 3, 3]), "x"), 0.0, abs_tol=1e-12),
            "un seul id répété : entropie nulle.", "un seul id répété doit donner une entropie nulle.")

In [ ]:
SIZES_20 = (300, 500, 1000, 2000, 4000, 8000)
if not bpe_written():
    print("⏳ Ex B2.20 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).")
else:
    with checked("B2.20"):
        key_20 = (mylearn.tokenizer.__file__, os.path.getmtime(mylearn.tokenizer.__file__))   # your file, as saved
        if globals().get("BIG_20_KEY") != key_20:
            start_20 = time.perf_counter()
            BIG_20 = mylearn.tokenizer.BPETokenizer(vocab_size=max(SIZES_20), pre_tokenizer="gpt2").fit(VERNE_TRAIN)
            BIG_20_KEY = key_20
            print(f"Un entraînement à {max(SIZES_20)} tokens sur VERNE_TRAIN : {fr(time.perf_counter() - start_20, 1)} s.")
        small_20 = mylearn.tokenizer.BPETokenizer(vocab_size=400, pre_tokenizer="gpt2").fit(VERNE_TRAIN)
        verdict("B2.20", truncated_bpe(BIG_20, 400).merges_ == small_20.merges_,
                f"les 144 premières fusions de l'entraînement à {max(SIZES_20)} tokens sont celles d'un entraînement à 400 : "
                "truncated_bpe redonne le tokenizer plus petit.",
                "les premières fusions d'un entraînement plus long devraient être celles d'un entraînement plus court : "
                "relis les conditions d'arrêt de fit (B2.16).")
        rows_20 = sweep_20(BIG_20, VERNE_VAL, SIZES_20)
        if not returned("B2.20", "sweep_20", rows_20):
            raise StopCheck
        table_20 = pd.DataFrame(rows_20)
        need_20 = ["vocab_size", "bytes_per_token", "bits_per_token", "bits_per_byte", "used"]
        if list(table_20.get("vocab_size", [])) != list(SIZES_20) or any(c not in table_20 for c in need_20):
            print(f"❌ Ex B2.20 : sweep_20 doit rendre une ligne par taille de {SIZES_20}, dans cet ordre, avec les clés {need_20}.")
            raise StopCheck
        print(table_20[need_20].to_string(index=False, float_format=lambda x: f"{x:.3f}"))
        fig_20, axes_20 = plt.subplots(1, 3, figsize=(13, 3.6))
        for ax, col, label in zip(axes_20, ("bytes_per_token", "bits_per_token", "bits_per_byte"),
                                  ("octets par token", "entropie (bits par token)", "bits par octet")):
            ax.plot(table_20["vocab_size"], table_20[col], "o-")
            ax.set_xscale("log")
            ax.set_xticks(SIZES_20, [str(s) for s in SIZES_20])
            ax.set_xlabel("taille du vocabulaire")
            ax.set_title(label)
            ax.grid(alpha=0.3)
        fig_20.suptitle("BPE octet de Verne (VERNE_TRAIN), mesuré sur VERNE_VAL")
        fig_20.tight_layout()
        plt.show()

### Ex B2.21 — Tokenizer entraîné sur Holmes, appliqué à Verne (et l'inverse) : le coût de la langue 🔬 ★★ ⏱️ 25 min
**Objectif :** mesurer ce que coûte un tokenizer appris sur une autre langue, et ce que coûte un tokenizer partagé.  
**Prérequis :** Ex B2.20 · Ex B2.13 · fiche §202.6 (« Le même texte, un autre prix selon la langue ») · **Fil rouge :** Holmes/Verne · **Parcours :** C

En B2.13, GPT-2 compresse moins bien Verne que Holmes. Cela vient-il du français, ou des textes sur lesquels son tokenizer a appris ? Tu entraînes trois BPE octet de 2 000 tokens et tu les mesures sur la validation des deux romans.

**Contrat**
- `train_tokenizers_21(vocab_size=2000)` : trois `BPETokenizer` de ta librairie, au niveau octet, de pré-tokenizer `'gpt2'` et de `vocab_size` tokens, dans un dictionnaire :
  - `"holmes"`, appris sur `HOLMES_TRAIN` ;
  - `"verne"`, appris sur `VERNE_TRAIN` ;
  - `"mixte"`, appris sur les deux textes ensemble.
- `cross_table_21(tokenizers, texts)` : un `DataFrame` de pandas des octets par token (`compression_20`, B2.20), une ligne par tokenizer et une colonne par texte ; `tokenizers` et `texts` sont des dictionnaires nom → objet.

La cellule de vérification entraîne tes trois tokenizers (de 3 s à 2 min selon ta façon de compter les paires) et contrôle leurs réglages. Elle vérifie aussi que celui de Verne commence par les fusions de B2.20. Elle calcule ensuite ton tableau, y ajoute GPT-2 et le trace.

Dans tes notes :
- de combien la compression baisse-t-elle quand on applique un tokenizer à l'autre roman ?
- le tokenizer mixte : que coûte-t-il à chaque langue, par rapport à son tokenizer « maison » ?
- où se place GPT-2, avec ses 50 257 tokens ?
- qu'en conclus-tu sur la « taxe » de certaines langues (fiche §202.6), et sur le choix des données d'entraînement du tokenizer d'un modèle multilingue ?

In [ ]:
def train_tokenizers_21(vocab_size=2000):
    """Three byte-level BPETokenizer of your library, pre_tokenizer "gpt2", vocab_size tokens:
    {"holmes": fitted on HOLMES_TRAIN, "verne": on VERNE_TRAIN, "mixte": on both texts}."""
    raise NotImplementedError("train_tokenizers_21")


def cross_table_21(tokenizers, texts):
    """A pandas DataFrame of the bytes per token (compression_20): one row per tokenizer, one column per text
    (tokenizers and texts are dicts name -> object)."""
    raise NotImplementedError("cross_table_21")

In [ ]:
if not bpe_written():
    print("⏳ Ex B2.21 : l'expérience attend BPETokenizer.fit et encode (B2.16 et B2.17).")
else:
    with checked("B2.21"):
        need("B2.21", "compression_20")
        start_21 = time.perf_counter()
        toks_21 = train_tokenizers_21(2000)
        print(f"Trois entraînements à 2 000 tokens : {fr(time.perf_counter() - start_21, 1)} s.")
        if not returned("B2.21", "train_tokenizers_21", toks_21):
            raise StopCheck
        if not isinstance(toks_21, dict) or set(toks_21) != {"holmes", "verne", "mixte"}:
            print("❌ Ex B2.21 : train_tokenizers_21 doit rendre un dictionnaire de clés 'holmes', 'verne' et 'mixte'.")
            raise StopCheck
        for name, tok in toks_21.items():
            ok = (getattr(tok, "level", None) == "byte" and getattr(tok, "pre_tokenizer", None) == "gpt2"
                  and len(getattr(tok, "vocab_", {})) == 2000 and not getattr(tok, "special_ids_", {}))
            verdict("B2.21", ok, f"{name} : BPE octet 'gpt2' de 2 000 tokens.",
                    f"{name} : attendu un BPETokenizer de niveau octet, pré-tokenizer 'gpt2', 2 000 tokens sans token spécial.")
        if "BIG_20" in globals():
            verdict("B2.21", toks_21["verne"].merges_ == BIG_20.merges_[:2000 - 256],
                    "le tokenizer de Verne a les 1 744 premières fusions de celui de B2.20 : il est bien appris sur VERNE_TRAIN.",
                    "le tokenizer de Verne devrait avoir les premières fusions de BIG_20 (B2.20) : sur quel texte est-il appris ?")
        verdict("B2.21", toks_21["holmes"].merges_[:20] != toks_21["verne"].merges_[:20]
                and toks_21["mixte"].merges_ not in (toks_21["holmes"].merges_, toks_21["verne"].merges_),
                "les trois tokenizers ont des fusions différentes.",
                "deux de tes tokenizers ont les mêmes fusions : sont-ils appris sur des textes différents ?")
        texts_21 = {"Holmes (validation)": HOLMES_VAL, "Verne (validation)": VERNE_VAL}
        table_21 = cross_table_21(toks_21, texts_21)
        if not returned("B2.21", "cross_table_21", table_21):
            raise StopCheck
        if not isinstance(table_21, pd.DataFrame) or set(table_21.index) != set(toks_21) or set(table_21.columns) != set(texts_21):
            print("❌ Ex B2.21 : cross_table_21 doit rendre un DataFrame, une ligne par tokenizer, une colonne par texte.")
            raise StopCheck
        verdict("B2.21", math.isclose(float(table_21.loc["verne", "Verne (validation)"]),
                                      compression_20(toks_21["verne"], VERNE_VAL)),
                "le tableau contient les octets par token de compression_20.",
                "le tableau doit contenir compression_20(tokenizer, texte) : relis-le.")
        gpt2_21 = wbtext.try_tokenizer("gpt2", "B2.21")
        if gpt2_21 is not None:
            table_21.loc["GPT-2 (50 257 tokens)"] = pd.Series({name: n_bytes(t) / wbtext.count_tokens(gpt2_21, t)
                                                              for name, t in texts_21.items()})
        print(table_21.to_string(float_format=lambda x: f"{x:.2f}"))
        ax_21 = table_21.plot.bar(figsize=(7.5, 3.6), rot=0)
        ax_21.set_ylabel("octets par token")
        ax_21.set_title("Chaque tokenizer sur la validation des deux textes")
        ax_21.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu compter les octets et les points de code d'un texte, et écrire un caractère en UTF-8 ?
2. Sais-tu dérouler un entraînement BPE à la main, départage compris, puis encoder un mot nouveau par rang ?
3. Sais-tu dire quelles lignes d'une table d'embedding reçoivent un gradient, et calculer une PPMI ?
4. Sais-tu chiffrer ce que coûtent une table d'embedding et un texte selon sa langue ?

**Pour aller plus loin** : le chapitre 6 du cours LLM de Hugging Face et `minbpe` d'A. Karpathy, cités dans la fiche. La suite : le bonus B3, attention et Transformers.

*Les exercices B2.22 à B2.31 (parties D et E : vecteurs de mots, embeddings de phrases, livraison et défi) seront ajoutés à la prochaine étape de génération ; relance alors `python tools/start_chapter.py B2` pour obtenir le notebook complet.*